In [0]:
CREATE OR REPLACE TEMPORARY VIEW seller_meio_pagamento AS (

WITH tb_pedidos AS (
  SELECT *
  FROM workspace.olist.orders
  WHERE order_purchase_timestamp < '2018-07-01'
),

tb_seller AS (
  SELECT
    order_id,
    seller_id
  FROM workspace.olist.order_items
  GROUP BY order_id, seller_id

),

tb_pagamentos AS (
select 
    order_id,
    SUM(CASE WHEN payment_type = 'credit_card' THEN payment_value ELSE 0 END) AS vlr_credit_card,
    SUM(CASE WHEN payment_type = 'boleto'      THEN payment_value ELSE 0 END) AS vlr_boleto,
    SUM(CASE WHEN payment_type = 'voucher'     THEN payment_value ELSE 0 END) AS vlr_voucher,
    SUM(CASE WHEN payment_type = 'debit_card'  THEN payment_value ELSE 0 END) AS vlr_debit_card,
    SUM(CASE WHEN payment_type not in ('credit_card', 'boleto', 'voucher', 'debit_card') THEN payment_value ELSE 0 END) AS vlr_outros

from workspace.olist.order_payments
GROUP by order_id
),

tb_base AS (
  SELECT
    s.seller_id,
    p.order_purchase_timestamp,
    pg.vlr_credit_card,
    pg.vlr_boleto,
    pg.vlr_voucher,
    pg.vlr_debit_card,
    pg.vlr_outros
  FROM tb_pedidos p
  INNER JOIN tb_seller     s  ON p.order_id = s.order_id
  LEFT JOIN tb_pagamentos pg ON p.order_id = pg.order_id
),
tb_estrutura AS (
  SELECT DISTINCT
    seller_id,
    DATE_TRUNC('month', order_purchase_timestamp)                       AS ref_month,
    DATE_FORMAT(DATE_TRUNC('month', order_purchase_timestamp), 'yyyyMM') AS ref_month_fmt
  FROM tb_base
)

SELECT
  sp.seller_id,
  sp.ref_month_fmt                                                        AS ref_month,

  -- D28
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28)  THEN b.vlr_credit_card END) AS avg_vlr_credit_card_d28,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28)  THEN b.vlr_boleto      END) AS avg_vlr_boleto_d28,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28)  THEN b.vlr_voucher     END) AS avg_vlr_voucher_d28,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28)  THEN b.vlr_debit_card  END) AS avg_vlr_debit_card_d28,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28)  THEN b.vlr_outros      END) AS avg_vlr_outros_d28,

  -- D56
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56)  THEN b.vlr_credit_card END) AS avg_vlr_credit_card_d56,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56)  THEN b.vlr_boleto      END) AS avg_vlr_boleto_d56,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56)  THEN b.vlr_voucher     END) AS avg_vlr_voucher_d56,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56)  THEN b.vlr_debit_card  END) AS avg_vlr_debit_card_d56,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56)  THEN b.vlr_outros      END) AS avg_vlr_outros_d56,

  -- D365
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_credit_card END) AS avg_vlr_credit_card_d365,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_boleto      END) AS avg_vlr_boleto_d365,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_voucher     END) AS avg_vlr_voucher_d365,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_debit_card  END) AS avg_vlr_debit_card_d365,
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_outros      END) AS avg_vlr_outros_d365,

  -- Vida
  AVG(b.vlr_credit_card) AS avg_vlr_credit_card_vida,
  AVG(b.vlr_boleto)      AS avg_vlr_boleto_vida,
  AVG(b.vlr_voucher)     AS avg_vlr_voucher_vida,
  AVG(b.vlr_debit_card)  AS avg_vlr_debit_card_vida,
  AVG(b.vlr_outros)      AS avg_vlr_outros_vida

FROM tb_estrutura sp
LEFT JOIN tb_base b
ON  b.seller_id = sp.seller_id
AND b.order_purchase_timestamp <  sp.ref_month


GROUP BY sp.seller_id, sp.ref_month, sp.ref_month_fmt
ORDER BY sp.seller_id, sp.ref_month

)

In [0]:
CREATE OR REPLACE TEMPORARY VIEW seller_parcelas AS (

WITH tb_pedidos AS (
  SELECT *
  FROM workspace.olist.orders
  WHERE order_purchase_timestamp < '2018-07-01'
),

tb_seller AS (
  SELECT
    order_id,
    seller_id
  FROM workspace.olist.order_items
  GROUP BY order_id, seller_id

),

tb_pagamentos AS (
select 
    order_id,
    max(payment_installments) AS max_payment_installments
    
from workspace.olist.order_payments
where payment_installments>0 
GROUP by order_id
),

tb_base AS (
  SELECT
    s.seller_id,
    p.order_purchase_timestamp,
    pg.max_payment_installments
  
  FROM tb_pedidos p
  INNER JOIN tb_seller     s  ON p.order_id = s.order_id
  LEFT JOIN tb_pagamentos pg ON p.order_id = pg.order_id
),
tb_estrutura AS (
  SELECT DISTINCT
    seller_id,
    DATE_TRUNC('month', order_purchase_timestamp)                       AS ref_month,
    DATE_FORMAT(DATE_TRUNC('month', order_purchase_timestamp), 'yyyyMM') AS ref_month_fmt
  FROM tb_base
)

SELECT
  sp.seller_id,
  sp.ref_month_fmt                                                        AS ref_month,

  -- D28
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28)  THEN b.max_payment_installments END) AS avg_payment_installments_d28,
 

  -- D56
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56)  THEN b.max_payment_installments END) AS avg_payment_installments_d56,


  -- D365
  AVG(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.max_payment_installments END) AS avg_payment_installments_d365,
 

  -- Vida
  AVG(b.max_payment_installments) AS avg_payment_installments_vida
 

FROM tb_estrutura sp
LEFT JOIN tb_base b
ON  b.seller_id = sp.seller_id
AND b.order_purchase_timestamp <  sp.ref_month


GROUP BY sp.seller_id, sp.ref_month, sp.ref_month_fmt
ORDER BY sp.seller_id, sp.ref_month
)


In [0]:
--Quantidade de Meios de pagamento vendidos pelo seller
--Feedback/Sugestão: 1 coluna por tipo de pagamento com % por tipo de pagamento (share)
--Share Qtde

CREATE OR REPLACE TEMPORARY VIEW  seller_share_qtde AS (

WITH tb_pedidos AS (
  SELECT *
  FROM workspace.olist.orders
  WHERE order_purchase_timestamp < '2018-07-01'
),

tb_seller AS (
  SELECT
    order_id,
    seller_id
  FROM workspace.olist.order_items
  GROUP BY order_id, seller_id

),

tb_pagamentos AS (
select 
    order_id,
    SUM(CASE WHEN payment_type = 'credit_card' THEN 1 ELSE 0 END) AS qtde_credit_card,
    SUM(CASE WHEN payment_type = 'boleto'     THEN 1 ELSE 0 END) AS qtde_boleto,
    SUM(CASE WHEN payment_type = 'voucher'     THEN 1 ELSE 0 END) AS qtde_voucher,
    SUM(CASE WHEN payment_type = 'debit_card'     THEN 1 ELSE 0 END) AS qtde_debit_card,
    SUM(CASE WHEN payment_type not in ('credit_card', 'boleto', 'voucher', 'debit_card') THEN 1 ELSE 0 END) AS qtde_outros
    
from workspace.olist.order_payments
GROUP by order_id
),

tb_base AS (
  SELECT
    s.seller_id,
    p.order_purchase_timestamp,
    pg.qtde_credit_card,
    pg.qtde_boleto,
    pg.qtde_voucher,
    pg.qtde_debit_card,
    pg.qtde_outros
  
  FROM tb_pedidos p
  INNER JOIN tb_seller     s  ON p.order_id = s.order_id
  LEFT JOIN tb_pagamentos pg ON p.order_id = pg.order_id
),
tb_estrutura AS (
  SELECT DISTINCT
    seller_id,
    DATE_TRUNC('month', order_purchase_timestamp)                       AS ref_month,
    DATE_FORMAT(DATE_TRUNC('month', order_purchase_timestamp), 'yyyyMM') AS ref_month_fmt
  FROM tb_base
)

SELECT
  sp.seller_id,
  sp.ref_month_fmt                                                        AS ref_month,

 -- D28
  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.qtde_credit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_credit_card_d28,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.qtde_boleto END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_boleto_d28,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.qtde_voucher END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_voucher_d28,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.qtde_debit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_debit_card_d28,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.qtde_outros END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_outros_d28,

  -- D56
  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.qtde_credit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_credit_card_d56,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.qtde_boleto END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_boleto_d56,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.qtde_voucher END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_voucher_d56,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.qtde_debit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_debit_card_d56,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.qtde_outros END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_outros_d56,

  -- D365
  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.qtde_credit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_credit_card_d365,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.qtde_boleto END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_boleto_d365,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.qtde_voucher END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_voucher_d365,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.qtde_debit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_debit_card_d365,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.qtde_outros END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros END), 0) AS share_qtde_outros_d365,

  -- Vida
  SUM(b.qtde_credit_card) /
  NULLIF(SUM(b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros), 0) AS share_qtde_credit_card_vida,

  SUM(b.qtde_boleto) /
  NULLIF(SUM(b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros), 0) AS share_qtde_boleto_vida,

  SUM(b.qtde_voucher) /
  NULLIF(SUM(b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros), 0) AS share_qtde_voucher_vida,

  SUM(b.qtde_debit_card) /
  NULLIF(SUM(b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros), 0) AS share_qtde_debit_card_vida,

  SUM(b.qtde_outros) /
  NULLIF(SUM(b.qtde_credit_card + b.qtde_boleto + b.qtde_voucher + b.qtde_debit_card + b.qtde_outros), 0) AS share_qtde_outros_vida

 

FROM tb_estrutura sp
LEFT JOIN tb_base b
ON  b.seller_id = sp.seller_id
AND b.order_purchase_timestamp <  sp.ref_month


GROUP BY sp.seller_id, sp.ref_month, sp.ref_month_fmt
ORDER BY sp.seller_id, sp.ref_month

)

In [0]:
--Quantidade de Meios de pagamento vendidos pelo seller
--Feedback/Sugestão: 1 coluna por tipo de pagamento com % por tipo de pagamento (share)
--Share Valor

create or replace temporary view seller_share_valor as (

WITH tb_pedidos AS (
  SELECT *
  FROM workspace.olist.orders
  WHERE order_purchase_timestamp < '2018-07-01'
),

tb_seller AS (
  SELECT
    order_id,
    seller_id
  FROM workspace.olist.order_items
  GROUP BY order_id, seller_id

),


tb_pagamentos AS (
select 
    order_id,
    SUM(CASE WHEN payment_type = 'credit_card' THEN payment_value ELSE 0 END) AS vlr_credit_card,
    SUM(CASE WHEN payment_type = 'boleto'      THEN payment_value ELSE 0 END) AS vlr_boleto,
    SUM(CASE WHEN payment_type = 'voucher'     THEN payment_value ELSE 0 END) AS vlr_voucher,
    SUM(CASE WHEN payment_type = 'debit_card'     THEN payment_value ELSE 0 END) AS vlr_debit_card,
    SUM(CASE WHEN payment_type not in ('credit_card', 'boleto', 'voucher', 'debit_card') THEN payment_value ELSE 0 END) AS vlr_outros

from workspace.olist.order_payments
GROUP by order_id
),

tb_base AS (
  SELECT
    s.seller_id,
    p.order_purchase_timestamp,
    pg.vlr_credit_card,
    pg.vlr_boleto,
    pg.vlr_voucher,
    pg.vlr_debit_card,
    pg.vlr_outros
  FROM tb_pedidos p
  INNER JOIN tb_seller     s  ON p.order_id = s.order_id
  LEFT JOIN tb_pagamentos pg ON p.order_id = pg.order_id
),
tb_estrutura AS (
  SELECT DISTINCT
    seller_id,
    DATE_TRUNC('month', order_purchase_timestamp)                       AS ref_month,
    DATE_FORMAT(DATE_TRUNC('month', order_purchase_timestamp), 'yyyyMM') AS ref_month_fmt
  FROM tb_base
)

SELECT
  sp.seller_id,
  sp.ref_month_fmt                                                        AS ref_month,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.vlr_credit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_credit_card_d28,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.vlr_boleto END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_boleto_d28,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.vlr_voucher END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_voucher_d28,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.vlr_debit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_debit_card_d28,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.vlr_outros END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 28) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_outros_d28,

  -- D56
  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.vlr_credit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_credit_card_d56,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.vlr_boleto END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_boleto_d56,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.vlr_voucher END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_voucher_d56,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.vlr_debit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_debit_card_d56,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.vlr_outros END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 56) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_outros_d56,

  -- D365
  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_credit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_credit_card_d365,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_boleto END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_boleto_d365,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_voucher END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_voucher_d365,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_debit_card END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_debit_card_d365,

  SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_outros END) /
  NULLIF(SUM(CASE WHEN b.order_purchase_timestamp >= DATE_SUB(sp.ref_month, 365) THEN b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros END), 0) AS share_valor_outros_d365,

  -- Vida
  SUM(b.vlr_credit_card) /
  NULLIF(SUM(b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros), 0) AS share_valor_credit_card_vida,

  SUM(b.vlr_boleto) /
  NULLIF(SUM(b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros), 0) AS share_valor_boleto_vida,

  SUM(b.vlr_voucher) /
  NULLIF(SUM(b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros), 0) AS share_valor_voucher_vida,

  SUM(b.vlr_debit_card) /
  NULLIF(SUM(b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros), 0) AS share_valor_debit_card_vida,

  SUM(b.vlr_outros) /
  NULLIF(SUM(b.vlr_credit_card + b.vlr_boleto + b.vlr_voucher + b.vlr_debit_card + b.vlr_outros), 0) AS share_valor_outros_vida

 

FROM tb_estrutura sp
LEFT JOIN tb_base b
ON  b.seller_id = sp.seller_id
AND b.order_purchase_timestamp <  sp.ref_month


GROUP BY sp.seller_id, sp.ref_month, sp.ref_month_fmt
ORDER BY sp.seller_id, sp.ref_month
)


In [0]:
create or replace temporary view all_table as (
select t1.*,
       t2.* EXCEPT (seller_id, ref_month),
       t3.* EXCEPT (seller_id, ref_month),
       t4.* EXCEPT (seller_id, ref_month)
from   seller_meio_pagamento t1
       left join seller_parcelas t2 on t1.seller_id = t2.seller_id and t1.ref_month = t2.ref_month
       left join seller_share_qtde t3 on t1.seller_id = t3.seller_id and t1.ref_month = t3.ref_month
       left join seller_share_valor t4 on t1.seller_id = t4.seller_id and t1.ref_month = t4.ref_month
);  

In [0]:
CREATE OR REPLACE TABLE workspace.olist.fs_seller_meio_pagamento (

  seller_id STRING
    COMMENT 'Identificador único do seller.',

  ref_month STRING
    COMMENT 'Mês de referência da feature no formato yyyyMM. Todas as métricas consideram apenas informações anteriores a este mês.',

  -- Valor médio por meio de pagamento
  avg_vlr_credit_card_d28 DOUBLE
    COMMENT 'Valor médio pago via cartão de crédito nos 28 dias anteriores ao mês de referência.',
  avg_vlr_boleto_d28 DOUBLE
    COMMENT 'Valor médio pago via boleto nos 28 dias anteriores ao mês de referência.',
  avg_vlr_voucher_d28 DOUBLE
    COMMENT 'Valor médio pago via voucher nos 28 dias anteriores ao mês de referência.',
  avg_vlr_debit_card_d28 DOUBLE
    COMMENT 'Valor médio pago via cartão de débito nos 28 dias anteriores ao mês de referência.',
  avg_vlr_outros_d28 DOUBLE
    COMMENT 'Valor médio pago por outros meios de pagamento nos 28 dias anteriores ao mês de referência.',
  avg_vlr_credit_card_d56 DOUBLE
    COMMENT 'Valor médio pago via cartão de crédito nos 56 dias anteriores ao mês de referência.',
  avg_vlr_boleto_d56 DOUBLE
    COMMENT 'Valor médio pago via boleto nos 56 dias anteriores ao mês de referência.',
  avg_vlr_voucher_d56 DOUBLE
    COMMENT 'Valor médio pago via voucher nos 56 dias anteriores ao mês de referência.',
  avg_vlr_debit_card_d56 DOUBLE
    COMMENT 'Valor médio pago via cartão de débito nos 56 dias anteriores ao mês de referência.',
  avg_vlr_outros_d56 DOUBLE
    COMMENT 'Valor médio pago por outros meios de pagamento nos 56 dias anteriores ao mês de referência.',
  avg_vlr_credit_card_d365 DOUBLE
    COMMENT 'Valor médio pago via cartão de crédito nos 365 dias anteriores ao mês de referência.',
  avg_vlr_boleto_d365 DOUBLE
    COMMENT 'Valor médio pago via boleto nos 365 dias anteriores ao mês de referência.',
  avg_vlr_voucher_d365 DOUBLE
    COMMENT 'Valor médio pago via voucher nos 365 dias anteriores ao mês de referência.',
  avg_vlr_debit_card_d365 DOUBLE
    COMMENT 'Valor médio pago via cartão de débito nos 365 dias anteriores ao mês de referência.',
  avg_vlr_outros_d365 DOUBLE
    COMMENT 'Valor médio pago por outros meios de pagamento nos 365 dias anteriores ao mês de referência.',
  avg_vlr_credit_card_vida DOUBLE
    COMMENT 'Valor médio histórico pago via cartão de crédito desde o início do histórico até o mês de referência.',
  avg_vlr_boleto_vida DOUBLE
    COMMENT 'Valor médio histórico pago via boleto desde o início do histórico até o mês de referência.',
  avg_vlr_voucher_vida DOUBLE
    COMMENT 'Valor médio histórico pago via voucher desde o início do histórico até o mês de referência.',
  avg_vlr_debit_card_vida DOUBLE
    COMMENT 'Valor médio histórico pago via cartão de débito desde o início do histórico até o mês de referência.',
  avg_vlr_outros_vida DOUBLE
    COMMENT 'Valor médio histórico pago por outros meios de pagamento desde o início do histórico até o mês de referência.',

  -- Parcelamento
  avg_payment_installments_d28 DOUBLE
    COMMENT 'Quantidade média de parcelas dos pagamentos nos 28 dias anteriores ao mês de referência.',
  avg_payment_installments_d56 DOUBLE
    COMMENT 'Quantidade média de parcelas dos pagamentos nos 56 dias anteriores ao mês de referência.',
  avg_payment_installments_d365 DOUBLE
    COMMENT 'Quantidade média de parcelas dos pagamentos nos 365 dias anteriores ao mês de referência.',
  avg_payment_installments_vida DOUBLE
    COMMENT 'Quantidade média histórica de parcelas dos pagamentos até o mês de referência.',

  -- Share de quantidade

  share_qtde_credit_card_d28 DOUBLE
    COMMENT 'Participação do cartão de crédito na quantidade de pagamentos dos últimos 28 dias.',
  share_qtde_boleto_d28 DOUBLE
    COMMENT 'Participação do boleto na quantidade de pagamentos dos últimos 28 dias.',
  share_qtde_voucher_d28 DOUBLE
    COMMENT 'Participação do voucher na quantidade de pagamentos dos últimos 28 dias.',
  share_qtde_debit_card_d28 DOUBLE
    COMMENT 'Participação do cartão de débito na quantidade de pagamentos dos últimos 28 dias.',
  share_qtde_outros_d28 DOUBLE
    COMMENT 'Participação de outros meios na quantidade de pagamentos dos últimos 28 dias.',
  share_qtde_credit_card_d56 DOUBLE
    COMMENT 'Participação do cartão de crédito na quantidade de pagamentos dos últimos 56 dias.',
  share_qtde_boleto_d56 DOUBLE
    COMMENT 'Participação do boleto na quantidade de pagamentos dos últimos 56 dias.',
  share_qtde_voucher_d56 DOUBLE
    COMMENT 'Participação do voucher na quantidade de pagamentos dos últimos 56 dias.',
  share_qtde_debit_card_d56 DOUBLE
    COMMENT 'Participação do cartão de débito na quantidade de pagamentos dos últimos 56 dias.',
  share_qtde_outros_d56 DOUBLE
    COMMENT 'Participação de outros meios na quantidade de pagamentos dos últimos 56 dias.',
  share_qtde_credit_card_d365 DOUBLE
    COMMENT 'Participação do cartão de crédito na quantidade de pagamentos dos últimos 365 dias.',
  share_qtde_boleto_d365 DOUBLE
    COMMENT 'Participação do boleto na quantidade de pagamentos dos últimos 365 dias.',
  share_qtde_voucher_d365 DOUBLE
    COMMENT 'Participação do voucher na quantidade de pagamentos dos últimos 365 dias.',
  share_qtde_debit_card_d365 DOUBLE
    COMMENT 'Participação do cartão de débito na quantidade de pagamentos dos últimos 365 dias.',
  share_qtde_outros_d365 DOUBLE
    COMMENT 'Participação de outros meios na quantidade de pagamentos dos últimos 365 dias.',
  share_qtde_credit_card_vida DOUBLE
    COMMENT 'Participação histórica do cartão de crédito na quantidade total de pagamentos.',
  share_qtde_boleto_vida DOUBLE
    COMMENT 'Participação histórica do boleto na quantidade total de pagamentos.',
  share_qtde_voucher_vida DOUBLE
    COMMENT 'Participação histórica do voucher na quantidade total de pagamentos.',
  share_qtde_debit_card_vida DOUBLE
    COMMENT 'Participação histórica do cartão de débito na quantidade total de pagamentos.',
  share_qtde_outros_vida DOUBLE
    COMMENT 'Participação histórica de outros meios na quantidade total de pagamentos.',

  -- Share de valor
  share_valor_credit_card_d28 DOUBLE
    COMMENT 'Participação do cartão de crédito no valor pago nos últimos 28 dias.',
  share_valor_boleto_d28 DOUBLE
    COMMENT 'Participação do boleto no valor pago nos últimos 28 dias.',
  share_valor_voucher_d28 DOUBLE
    COMMENT 'Participação do voucher no valor pago nos últimos 28 dias.',
  share_valor_debit_card_d28 DOUBLE
    COMMENT 'Participação do cartão de débito no valor pago nos últimos 28 dias.',
  share_valor_outros_d28 DOUBLE
    COMMENT 'Participação de outros meios no valor pago nos últimos 28 dias.',
  share_valor_credit_card_d56 DOUBLE
    COMMENT 'Participação do cartão de crédito no valor pago nos últimos 56 dias.',
  share_valor_boleto_d56 DOUBLE
    COMMENT 'Participação do boleto no valor pago nos últimos 56 dias.',
  share_valor_voucher_d56 DOUBLE
    COMMENT 'Participação do voucher no valor pago nos últimos 56 dias.',
  share_valor_debit_card_d56 DOUBLE
    COMMENT 'Participação do cartão de débito no valor pago nos últimos 56 dias.',
  share_valor_outros_d56 DOUBLE
    COMMENT 'Participação de outros meios no valor pago nos últimos 56 dias.',
  share_valor_credit_card_d365 DOUBLE
    COMMENT 'Participação do cartão de crédito no valor pago nos últimos 365 dias.',
  share_valor_boleto_d365 DOUBLE
    COMMENT 'Participação do boleto no valor pago nos últimos 365 dias.',
  share_valor_voucher_d365 DOUBLE
    COMMENT 'Participação do voucher no valor pago nos últimos 365 dias.',
  share_valor_debit_card_d365 DOUBLE
    COMMENT 'Participação do cartão de débito no valor pago nos últimos 365 dias.',
  share_valor_outros_d365 DOUBLE
    COMMENT 'Participação de outros meios no valor pago nos últimos 365 dias.',
  share_valor_credit_card_vida DOUBLE
    COMMENT 'Participação histórica do cartão de crédito no valor total pago.',
  share_valor_boleto_vida DOUBLE
    COMMENT 'Participação histórica do boleto no valor total pago.',
  share_valor_voucher_vida DOUBLE
    COMMENT 'Participação histórica do voucher no valor total pago.',
  share_valor_debit_card_vida DOUBLE
    COMMENT 'Participação histórica do cartão de débito no valor total pago.',
  share_valor_outros_vida DOUBLE
    COMMENT 'Participação histórica de outros meios no valor total pago.'

)
COMMENT 'Feature Store de sellers contendo métricas históricas de comportamento de pagamento. A granularidade da tabela é seller_id e ref_month. Todas as features utilizam apenas informações anteriores ao mês de referência para evitar vazamento de dados.'
;

In [0]:
INSERT OVERWRITE workspace.olist.fs_seller_meio_pagamento
select * from all_table